# Transcripción de Audio con Whisper

## Paso 1: Instalación de librerías

In [ ]:
# Instalamos la librería de Whisper desde el repositorio oficial
# !pip install git+https://github.com/openai/whisper.git

# Aseguramos que tenemos ffmpeg (necesario para procesar audio)
# !sudo apt update && sudo apt install ffmpeg -y
# %pip install imageio-ffmpeg 

%pip install imageio-ffmpeg

Note: you may need to restart the kernel to use updated packages.


## Paso 2: Importar librerías y configurar el modelo

In [36]:
import os
import stat
import shutil
from pathlib import Path
import imageio_ffmpeg

# Ruta real del binario descargado por imageio-ffmpeg
src = Path(imageio_ffmpeg.get_ffmpeg_exe())
print("Binario real:", src)

# Carpeta de usuario sin permisos de admin
user_bin = Path.home() / ".local" / "bin"
user_bin.mkdir(parents=True, exist_ok=True)

# Crear un enlace llamado exactamente "ffmpeg"
dst = user_bin / "ffmpeg"

if dst.exists() or dst.is_symlink():
    dst.unlink()

dst.symlink_to(src)

# Asegurar permisos de ejecución por si acaso
dst.chmod(dst.stat().st_mode | stat.S_IEXEC)

# Añadir al PATH para esta sesión
os.environ["PATH"] = str(user_bin) + os.pathsep + os.environ.get("PATH", "")

print("Enlace creado:", dst)
print("ffmpeg en PATH:", shutil.which("ffmpeg"))

Binario real: /home/ciabd14/anaconda3/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-linux-x86_64-v7.0.2
Enlace creado: /home/ciabd14/.local/bin/ffmpeg
ffmpeg en PATH: /home/ciabd14/.local/bin/ffmpeg


In [37]:
import subprocess
subprocess.run(["ffmpeg", "-version"], check=True)

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
built with gcc 8 (Debian 8.3.0-6)
configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
libavutil      59.  8.100 / 59.  8.100
libavcodec     61.  3.100 / 61.  3.100
libavformat    61.  1.10

CompletedProcess(args=['ffmpeg', '-version'], returncode=0)

In [38]:
import whisper
import os

# Elegimos el modelo (puedes cambiar "base" por "small" o "medium" según tu PC/GPU)
model = whisper.load_model("base")
print("Modelo cargado correctamente.")

Modelo cargado correctamente.


## Paso 3: Cargar el archivo de audio

In [39]:
audio_path = "informatica.mp3" # Cambia esto por el nombre de tu archivo

if not os.path.exists(audio_path):
    print(f"Error: El archivo {audio_path} no existe. Por favor, súbelo.")
else:
    print(f"Archivo {audio_path} listo para procesar.")

Archivo informatica.mp3 listo para procesar.


## Paso 4: Realizar la transcripción

In [40]:
# Transcribir el audio
result = model.transcribe(audio_path, verbose=False)

# El resultado es un diccionario que contiene el texto y otros metadatos
print("-" * 30)
print("TRANSCRIPCIÓN FINALIZADA")
print("-" * 30)
print(result["text"])

Detected language: Spanish


100%|██████████| 2459/2459 [00:01<00:00, 1486.53frames/s]

------------------------------
TRANSCRIPCIÓN FINALIZADA
------------------------------
 La informática transforma en el mundo en pocas necanas. Gracias a los microprocesadores, hoy llevamos en el bolsillo más potencia de calculo que la usana para llargar la luna. Los saludos y el software están detrás de casi todo, desneras redes sociales hasta los diagnósticos médicos. Sin duda, la informática es el motor invisible en la sociedad monera.


## Paso 5: Guardar la transcripción en un archivo de texto

In [41]:
nombre_salida = "transcripcion_resultado.txt"

with open(nombre_salida, "w", encoding="utf-8") as f:
    f.write(result["text"])
    
print(f"La transcripción se ha guardado en: {nombre_salida}")

La transcripción se ha guardado en: transcripcion_resultado.txt


# Opciones Avanzadas

## A) Transcripción con marcas de tiempo (Timestamps)

In [42]:
for segment in result['segments']:
    start = segment['start']
    end = segment['end']
    text = segment['text']
    print(f"[{start:5.2f}s -> {end:5.2f}s] {text}")

[ 0.00s ->  4.00s]  La informática transforma en el mundo en pocas necanas.
[ 4.00s -> 11.70s]  Gracias a los microprocesadores, hoy llevamos en el bolsillo más potencia de calculo que la usana para llargar la luna.
[11.70s -> 15.44s]  Los saludos y el software están detrás de casi todo,
[15.44s -> 19.50s]  desneras redes sociales hasta los diagnósticos médicos.
[19.50s -> 24.50s]  Sin duda, la informática es el motor invisible en la sociedad monera.


## B) Traducir automáticamente al inglés

In [43]:
# Usamos el parámetro task="translate"
result_en = model.transcribe(audio_path, task="translate")
print(result_en["text"])

 Informatics transform the world into small companies. Thanks to the microprofaces, today we go to the most powerful volcano in Calculo that the Ussana para niega la luna. The satellites and the software are behind almost everything, the networks and the social networks are the news media. Without one, the informatics is the invisible motor in the social network.


## C) Especificar el idioma

In [44]:
result = model.transcribe(audio_path, language="es")